# Rolling Shutter Investigation — Direct Arrow Access

This notebook investigates the rolling shutter offset and LiDAR timestamp interpretation
for nuPlan by reading Arrow files directly — no SceneAPI, no sync table.

**Advantages over the SceneAPI version:**
- Uses **all 8800 ego poses** at 100 Hz (vs 80 sync-aligned poses at 20 Hz)
- Uses the **static camera-to-IMU transform** from Arrow schema metadata
- Filters for the **top lidar only** from the merged point cloud
- No sync table dependency — works with raw modality timestamps

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pyarrow import ipc
from scipy.spatial.transform import Rotation, Slerp

from py123d.common.io.camera.jpeg_camera_io import load_image_from_jpeg_file
from py123d.datatypes.sensors.lidar import LidarID
from py123d.datatypes.sensors.pinhole_camera import PinholeCameraMetadata
from py123d.parser.nuplan.nuplan_sensor_io import load_nuplan_point_cloud_data_from_path
from py123d.visualization.matplotlib.helper import undistort_image_from_camera

## 1. Configuration

In [ ]:
LOG_DIR = Path("/home/daniel/py123d_workspace/data/logs/nuplan-mini_train/2021.10.11.07.12.18_veh-50_00211_00304")
SENSOR_ROOT = Path("/media/nvme1/nuplan/dataset/nuplan-v1.1/sensor_blobs")

CAMERA_NAME = "pcam_l1"  # Front camera
FRAME_INDEX = 786 // 2  # Which camera frame to use
LIDAR_SWEEP_DURATION_US = 50_000  # 50 ms for 20 Hz spinning LiDAR

# Matching strategy: "forward" (lidar_ts <= cam_ts) or "backward" (lidar_ts >= cam_ts)
# If lidar timestamp = start of sweep: use "forward" (sweep starts before camera captures)
# If lidar timestamp = end of sweep: use "backward" (sweep ends after camera captures)
LIDAR_MATCH_MODE = "forward"

## 2. Load Arrow Tables

In [ ]:
def read_arrow_table(path: Path):
    """Read an Arrow IPC file."""
    with ipc.open_file(path) as reader:
        return reader.read_all()


ego_table = read_arrow_table(LOG_DIR / "ego_state_se3.arrow")
cam_table = read_arrow_table(LOG_DIR / f"pinhole_camera.{CAMERA_NAME}.arrow")
lidar_table = read_arrow_table(LOG_DIR / "lidar.lidar_merged.arrow")

print(f"Ego state:  {ego_table.num_rows} rows (full log, 100 Hz)")
print(f"Camera:     {cam_table.num_rows} rows")
print(f"LiDAR:      {lidar_table.num_rows} rows")

## 3. Extract Static Camera-to-IMU Transform from Schema Metadata

In [ ]:
# Camera metadata is stored in the Arrow schema metadata (static, per-log)
cam_meta_dict = json.loads(cam_table.schema.metadata[b"metadata"])
cam_metadata = PinholeCameraMetadata.from_dict(cam_meta_dict)

# Static camera-to-IMU extrinsic (4x4 homogeneous matrix)
T_imu_from_cam = cam_metadata.camera_to_imu_se3.transformation_matrix
K = cam_metadata.intrinsics.camera_matrix

print(f"Camera: {cam_metadata.camera_name}")
print(f"Resolution: {cam_metadata.width} x {cam_metadata.height}")
print(
    f"Camera-to-IMU translation: {cam_metadata.camera_to_imu_se3.x:.3f}, {cam_metadata.camera_to_imu_se3.y:.3f}, {cam_metadata.camera_to_imu_se3.z:.3f}"
)

## 4. Build Ego Pose Interpolator from All Ego Timestamps

Using the full ego table (100 Hz) instead of sync-aligned samples gives us
much better interpolation accuracy — 10ms spacing vs 50ms.

In [ ]:
# Extract all ego poses directly from Arrow columns
ego_timestamps_us = np.array(ego_table["ego_state_se3.timestamp_us"].to_pylist(), dtype=np.float64)
ego_poses_raw = np.array(ego_table["ego_state_se3.imu_se3"].to_pylist(), dtype=np.float64)  # (N, 7): x,y,z,qw,qx,qy,qz

ego_translations = ego_poses_raw[:, :3]
# Arrow stores (qw, qx, qy, qz), scipy expects (qx, qy, qz, qw)
ego_quaternions = ego_poses_raw[:, [4, 5, 6, 3]]

# Build SLERP interpolator
ego_rotations = Rotation.from_quat(ego_quaternions)
slerp = Slerp(ego_timestamps_us, ego_rotations)

print(f"Ego poses: {len(ego_timestamps_us)} samples at {1e6 / np.diff(ego_timestamps_us).mean():.0f} Hz")
print(f"Time range: {(ego_timestamps_us[-1] - ego_timestamps_us[0]) / 1e6:.1f} s")
print(f"dt between poses: {np.diff(ego_timestamps_us).mean() / 1e3:.1f} ms")


def interpolate_ego_pose(query_us: float) -> np.ndarray:
    """Interpolate the ego IMU pose at an arbitrary timestamp. Returns 4x4 matrix."""
    query_us = np.clip(query_us, ego_timestamps_us[0], ego_timestamps_us[-1])
    tx = np.interp(query_us, ego_timestamps_us, ego_translations[:, 0])
    ty = np.interp(query_us, ego_timestamps_us, ego_translations[:, 1])
    tz = np.interp(query_us, ego_timestamps_us, ego_translations[:, 2])
    rot = slerp(query_us)
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = rot.as_matrix()
    T[:3, 3] = [tx, ty, tz]
    return T

## 5. Match and Load Camera-LiDAR Pairs

Camera (~10 Hz) and LiDAR (~20 Hz) have different rates. We pick a camera frame,
then find the matching LiDAR sweep using forward or backward timestamp matching.

In [ ]:
# Extract all timestamps as arrays
cam_all_ts = np.array(cam_table[f"pinhole_camera.{CAMERA_NAME}.timestamp_us"].to_pylist(), dtype=np.int64)
lidar_all_ts = np.array(lidar_table["lidar.lidar_merged.start_timestamp_us"].to_pylist(), dtype=np.int64)

print(f"Camera: {len(cam_all_ts)} frames at ~{1e6 / np.diff(cam_all_ts).mean():.0f} Hz")
print(f"LiDAR:  {len(lidar_all_ts)} frames at ~{1e6 / np.diff(lidar_all_ts).mean():.0f} Hz")


def find_lidar_index(cam_timestamp_us: int, mode: str = "forward") -> int:
    """Find the lidar frame matching a camera timestamp.

    :param cam_timestamp_us: Camera timestamp in microseconds.
    :param mode: Matching strategy:
        - "forward": lidar_ts <= cam_ts (lidar timestamp is start of interval ending at cam_ts)
        - "backward": lidar_ts >= cam_ts (lidar timestamp is end of interval starting at cam_ts)
    :return: Index into lidar_all_ts.
    """
    idx = np.searchsorted(lidar_all_ts, cam_timestamp_us)
    if mode == "forward":
        # Last lidar timestamp <= camera timestamp
        return max(0, idx - 1) if idx >= len(lidar_all_ts) or lidar_all_ts[idx] > cam_timestamp_us else idx
    elif mode == "backward":
        # First lidar timestamp >= camera timestamp
        return min(idx, len(lidar_all_ts) - 1)
    else:
        raise ValueError(f"Unknown mode: {mode}")


# Show matching for a few frames
CAM_FRAME = FRAME_INDEX
for mode in ["forward", "backward"]:
    lid_idx = find_lidar_index(cam_all_ts[CAM_FRAME], mode=mode)
    delta_ms = (cam_all_ts[CAM_FRAME] - lidar_all_ts[lid_idx]) / 1e3
    print(
        f"\n{mode:>8} match: cam[{CAM_FRAME}]={cam_all_ts[CAM_FRAME]} -> lidar[{lid_idx}]={lidar_all_ts[lid_idx]}  (delta={delta_ms:+.1f} ms)"
    )

In [ ]:
# Load the matched camera-lidar pair
LIDAR_FRAME = find_lidar_index(cam_all_ts[CAM_FRAME], mode=LIDAR_MATCH_MODE)

cam_ts_us = int(cam_all_ts[CAM_FRAME])
lidar_ts_us = int(lidar_all_ts[LIDAR_FRAME])

# Camera image
cam_image_path = SENSOR_ROOT / cam_table[f"pinhole_camera.{CAMERA_NAME}.data"][CAM_FRAME].as_py()
image_raw = load_image_from_jpeg_file(cam_image_path)
image = undistort_image_from_camera(image_raw.copy(), cam_metadata, mode="keep_focal_length")
img_shape = image.shape[:2]

# LiDAR: load merged point cloud, filter to top lidar only
lidar_path = SENSOR_ROOT / lidar_table["lidar.lidar_merged.data"][LIDAR_FRAME].as_py()
point_cloud_3d, point_cloud_features = load_nuplan_point_cloud_data_from_path(lidar_path)
top_lidar_mask = point_cloud_features["ids"] == int(LidarID.LIDAR_TOP)
lidar_xyz = point_cloud_3d[top_lidar_mask].astype(np.float64)

print(f"Match mode:       {LIDAR_MATCH_MODE}")
print(f"Camera frame:     {CAM_FRAME} (ts={cam_ts_us})")
print(f"LiDAR frame:      {LIDAR_FRAME} (ts={lidar_ts_us})")
print(f"Cam-LiDAR delta:  {(cam_ts_us - lidar_ts_us) / 1e3:.1f} ms")
print(f"Image shape:      {image.shape}")
print(f"Top lidar points: {lidar_xyz.shape[0]} / {point_cloud_3d.shape[0]} total")

In [ ]:
# Estimate ego velocity
idx = np.searchsorted(ego_timestamps_us, cam_ts_us)
idx = min(idx, len(ego_timestamps_us) - 2)
dt_s = (ego_timestamps_us[idx + 1] - ego_timestamps_us[idx]) / 1e6
vel = np.linalg.norm(ego_translations[idx + 1] - ego_translations[idx]) / dt_s
print(f"Ego velocity: ~{vel:.1f} m/s ({vel * 3.6:.1f} km/h)")
print(f"  -> 1 ms error  = ~{vel * 0.001 * 100:.1f} cm displacement")
print(f"  -> 17 ms error = ~{vel * 0.017 * 100:.1f} cm displacement")

## 6. Projection and Visualization Helpers

In [ ]:
def project_lidar_to_camera(
    lidar_xyz: np.ndarray,
    T_global_imu_lidar: np.ndarray,
    T_global_imu_camera: np.ndarray,
    camera_to_imu_matrix: np.ndarray,
    K: np.ndarray,
    img_shape: tuple,
    eps: float = 1e-3,
) -> tuple:
    """Project LiDAR points (in IMU frame) to camera pixels."""
    N = lidar_xyz.shape[0]
    points_h = np.concatenate([lidar_xyz, np.ones((N, 1))], axis=1).astype(np.float64)

    # IMU(t_lidar) -> global -> IMU(t_camera) -> camera
    T_cam_from_imu = np.linalg.inv(camera_to_imu_matrix)
    T_full = T_cam_from_imu @ np.linalg.inv(T_global_imu_camera) @ T_global_imu_lidar

    points_cam = (T_full @ points_h.T).T[:, :3]
    depths = points_cam[:, 2]
    valid = depths > eps

    pixels = (K @ points_cam.T).T
    pixels_uv = pixels[:, :2] / np.maximum(pixels[:, 2:3], eps)

    img_h, img_w = img_shape
    valid &= (pixels_uv[:, 0] > 0) & (pixels_uv[:, 0] < img_w - 1)
    valid &= (pixels_uv[:, 1] > 0) & (pixels_uv[:, 1] < img_h - 1)

    return pixels_uv, valid, depths


def overlay_lidar_on_image(image, pixels_uv, valid, depths, point_size=3):
    """Overlay LiDAR points on image, colored by depth."""
    img = image.copy()
    uv = pixels_uv[valid]
    d = depths[valid]
    d_clip = np.clip(d, 1.0, np.percentile(d, 95))
    d_norm = ((d_clip - d_clip.min()) / (d_clip.max() - d_clip.min() + 1e-6) * 255).astype(np.uint8)
    colors = cv2.applyColorMap(d_norm.reshape(-1, 1), cv2.COLORMAP_TURBO).reshape(-1, 3)
    for (x, y), color in zip(uv.astype(int), colors):
        cv2.circle(img, (x, y), point_size, (int(color[0]), int(color[1]), int(color[2])), -1)
    return img

## 7. Projection Grid: LiDAR Timestamp x Camera Rolling Shutter Offset

In [ ]:
lidar_offset_hypotheses = {
    "lidar_ts=start": 0,
    "lidar_ts=mid": -LIDAR_SWEEP_DURATION_US // 2,
    "lidar_ts=end": -LIDAR_SWEEP_DURATION_US,
}

camera_rs_offsets_us = {
    "cam_rs=-25ms": -25_000,
    "cam_rs=-17ms": -16_667,
    "cam_rs=-12ms": -12_000,
    "cam_rs=-8ms": -8_000,
    "cam_rs=0ms": 0,
    "cam_rs=8ms": 8_000,
    "cam_rs=12ms": 12_000,
    "cam_rs=17ms": 16_667,
    "cam_rs=22ms": 21_666,
}

n_lidar = len(lidar_offset_hypotheses)
n_cam = len(camera_rs_offsets_us)

fig, axes = plt.subplots(n_lidar, n_cam, figsize=(6 * n_cam, 4 * n_lidar))
if n_lidar == 1:
    axes = axes[np.newaxis, :]

for row, (lidar_label, lidar_offset_us) in enumerate(lidar_offset_hypotheses.items()):
    lidar_ego_us = lidar_ts_us + lidar_offset_us  # + LIDAR_SWEEP_DURATION_US // 2
    T_global_imu_lidar = interpolate_ego_pose(lidar_ego_us)

    for col, (cam_label, cam_rs_us) in enumerate(camera_rs_offsets_us.items()):
        cam_ego_us = cam_ts_us + cam_rs_us
        T_global_imu_cam = interpolate_ego_pose(cam_ego_us)

        pixels_uv, valid, depths = project_lidar_to_camera(
            lidar_xyz,
            T_global_imu_lidar,
            T_global_imu_cam,
            T_imu_from_cam,
            K,
            img_shape,
        )

        overlay = overlay_lidar_on_image(image, pixels_uv, valid, depths, point_size=3)

        dt_ms = (cam_ego_us - lidar_ego_us) / 1e3
        axes[row, col].imshow(overlay)
        axes[row, col].set_title(f"{lidar_label} | {cam_label}\ndt={dt_ms:.1f}ms", fontsize=9)
        axes[row, col].axis("off")

plt.suptitle(
    f"LiDAR-to-Camera Projection (top lidar, 100Hz ego, match={LIDAR_MATCH_MODE})",
    fontsize=13,
    y=1.01,
)
fig.savefig("07_rolling_shutter_direct_arrow.pdf", bbox_inches="tight", dpi=300)
plt.tight_layout()
plt.show()

## 8. Fine Sweep: Camera Offset Only

In [ ]:
# Assume lidar timestamp = start of sweep (adjust based on grid results above)
LIDAR_TS_IS_START = True
lidar_offset_us = 0 if LIDAR_TS_IS_START else -LIDAR_SWEEP_DURATION_US
lidar_ego_us = lidar_ts_us + lidar_offset_us + LIDAR_SWEEP_DURATION_US // 2
T_global_imu_lidar = interpolate_ego_pose(lidar_ego_us)

cam_offsets_ms = np.arange(-5, 35, 2.5)

n_cols = 4
n_rows = int(np.ceil(len(cam_offsets_ms) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, offset_ms in enumerate(cam_offsets_ms):
    cam_ego_us = cam_ts_us + offset_ms * 1000
    T_global_imu_cam = interpolate_ego_pose(cam_ego_us)

    pixels_uv, valid, depths = project_lidar_to_camera(
        lidar_xyz,
        T_global_imu_lidar,
        T_global_imu_cam,
        T_imu_from_cam,
        K,
        img_shape,
    )

    overlay = overlay_lidar_on_image(image, pixels_uv, valid, depths, point_size=4)
    axes[i].imshow(overlay)
    axes[i].set_title(f"RS offset = {offset_ms:.1f} ms", fontsize=9)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(
    f"Fine Camera RS Offset Sweep (top lidar, 100Hz ego, match={LIDAR_MATCH_MODE})",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
fig.savefig("08_rolling_shutter_direct_arrow_fine_sweep.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 9. Multi-Frame Validation

In [ ]:
FRAMES_TO_TEST = list(range(2, min(cam_table.num_rows, 10)))
OFFSETS_MS = [0, 8, 17]

n_frames = len(FRAMES_TO_TEST)
n_offsets = len(OFFSETS_MS)

fig, axes = plt.subplots(n_frames, n_offsets, figsize=(5 * n_offsets, 3.5 * n_frames))
if n_frames == 1:
    axes = axes[np.newaxis, :]

for row, cam_idx in enumerate(FRAMES_TO_TEST):
    # Match lidar to this camera frame
    lid_idx = find_lidar_index(cam_all_ts[cam_idx], mode=LIDAR_MATCH_MODE)

    # Load camera
    cam_ts_it = int(cam_all_ts[cam_idx])
    cam_path_it = SENSOR_ROOT / cam_table[f"pinhole_camera.{CAMERA_NAME}.data"][cam_idx].as_py()
    img_it = undistort_image_from_camera(
        load_image_from_jpeg_file(cam_path_it).copy(), cam_metadata, mode="keep_focal_length"
    )

    # Load lidar (top only)
    lidar_ts_it = int(lidar_all_ts[lid_idx])
    lidar_path_it = SENSOR_ROOT / lidar_table["lidar.lidar_merged.data"][lid_idx].as_py()
    pc3d_it, pcf_it = load_nuplan_point_cloud_data_from_path(lidar_path_it)
    top_mask = pcf_it["ids"] == int(LidarID.LIDAR_TOP)
    xyz_it = pc3d_it[top_mask].astype(np.float64)

    lidar_ego_it = lidar_ts_it + lidar_offset_us + LIDAR_SWEEP_DURATION_US // 2
    T_lidar_it = interpolate_ego_pose(lidar_ego_it)

    for col, offset_ms in enumerate(OFFSETS_MS):
        cam_ego_it = cam_ts_it + offset_ms * 1000
        T_cam_it = interpolate_ego_pose(cam_ego_it)

        pix, val, dep = project_lidar_to_camera(
            xyz_it,
            T_lidar_it,
            T_cam_it,
            T_imu_from_cam,
            K,
            img_it.shape[:2],
        )

        overlay = overlay_lidar_on_image(img_it, pix, val, dep, point_size=2)
        axes[row, col].imshow(overlay)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"RS = {offset_ms} ms", fontsize=10)
    axes[row, 0].set_ylabel(f"cam[{cam_idx}]", fontsize=9, rotation=0, labelpad=40)

plt.suptitle(f"Multi-Frame Comparison (top lidar, 100Hz ego, match={LIDAR_MATCH_MODE})", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig("09_rolling_shutter_direct_arrow_multi_frame.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 10. Baseline: Same Ego Pose (No Timing Correction)

In [ ]:
# Use the camera timestamp for both lidar and camera ego pose
T_same = interpolate_ego_pose(cam_ts_us)

pixels_uv_bl, valid_bl, depths_bl = project_lidar_to_camera(
    lidar_xyz,
    T_same,
    T_same,
    T_imu_from_cam,
    K,
    img_shape,
)

overlay_bl = overlay_lidar_on_image(image, pixels_uv_bl, valid_bl, depths_bl, point_size=3)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(overlay_bl)
ax.set_title("Baseline: same ego pose for LiDAR & camera (no timing correction)", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Timestamp Analysis

Print the raw timestamps to understand the relationship between ego, camera, and lidar timing.

In [ ]:
# Show matched camera-lidar pairs for both modes
print(f"{'cam_idx':>7}  {'cam_ts':>18}  {'mode':>8}  {'lid_idx':>7}  {'lidar_ts':>18}  {'delta_ms':>10}")
print("-" * 90)
for i in range(min(10, len(cam_all_ts))):
    ct = cam_all_ts[i]
    for mode in ["forward", "backward"]:
        lid_idx = find_lidar_index(ct, mode=mode)
        lt = lidar_all_ts[lid_idx]
        delta = (ct - lt) / 1e3
        print(f"{i:>7}  {ct:>18}  {mode:>8}  {lid_idx:>7}  {lt:>18}  {delta:>+9.1f} ms")